<div style="
background:#EEF2FF;
padding:20px;
border-left:8px solid #4F46E5;
border-radius:10px;">

<h2 style="color:#312E81;margin-top:0;">
🎬 Time Series Analysis and Forecasting of Movie Genre Popularity
</h2>
</div>

**Name:** `Hemant Kumar (19)` <br>
**Program:** `Master of Computer Applications (MCA)` <br>
**Subject:** `Time Series Data Analysis` <br>
**Dataset** `MovieLens 32M` - [Dataset Link](https://grouplens.org/datasets/movielens/32m/)  
**Research Paper** `Shahid & Islam (2023), PeerJ CS` [Paper Link](https://peerj.com/articles/cs-1603/)

---

## **Project Objective**

**Problem Definition**  
`This project aims to analyze the historical time-series data of movie genre popularity (1990-2023) and determine whether these patterns can be used to forecast future popularity.`

The study focuses on the following genres:

- Action
- Adventure
- Comedy
- Drama
- Romance
- Sci-Fi
- Thriller <br>
`although there were 19 genres in total, I PERSONALLY decided to work with the biggest/top 7`

---

**Applications**  
- TREND RELATED - production houses can use this information to decide which genre of movies to produce in the future. which will help them to maximize their revenue and also useful to minimize their chances of producing a movie that will not be suitable for future release.
- SEASONALITY RELATED - certain genre are more popular during certain seasons. for example horror movies during halloween, romantic movies during valentine's day, etc. so it can help in deciding which genre movies to schedule releases during that season.
- Time series forecast can be used to plan marketing campaigns, in case if for an upcoming movie the popularity is decreasing or increasing, then the campaign and promotion can be reduced or extended.

---

## Methodology

The project is divided into three major phases as notebooks:

### Phase I: Raw Data Loading and Preprocessing
- Load the MovieLens 32M dataset (movies & ratings)
- Brief Overview and Preprocessing
- Constructing final time series data for experiments

### Phase II: Exploratory Data Analysis (EDA)

- Data Inspection
- Missing Value Analysis
- Trend Analysis
- Seasonal Pattern Detection
- Stationarity Testing
- Correlation Analysis
- Time Series Visualization

### Phase II: Forecasting Models & Comparisons

- Naive*
- ARIMA
- SARIMA
- ETS

---

## Evaluation Metrics

The predictive performance of all forecasting models will be evaluated using:

- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- MAPE (Mean Absolute Percentage Error)

---

### Dataset Period

**March 1996 – October 2023**

What is Popularity in our Project?
- Monthly observations representing the distribution of movie genre popularity over time.
- It can also be seen as total rating activity per genre over the monthly time.

In [20]:
import pandas as pd
#import matplotlib.pyplot as plt

<div style="
background:#1F2937;
padding:12px;
border-left:8px solid #3B82F6;
border-radius:8px;
color:white;
font-size:24px;
font-weight:bold;">

💾 Section 1: Loading Raw Movies Metadata

</div>

In [21]:
print("[1/5] Loading movies metadata...\n")

movies_path = "../data/movielens32m/movies.csv"
movies_df = pd.read_csv(movies_path)

print(f"Loaded {len(movies_df):,} movies.")
movies_df.head(10) # display better

[1/5] Loading movies metadata...

Loaded 87,585 movies.


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


<div style="
background:#1F2937;
padding:12px;
border-left:8px solid #3B82F6;
border-radius:8px;
color:white;
font-size:24px;
font-weight:bold;">

💾 Section 2: Loading Raw Ratings data

</div>

In [22]:
print("\n[2/5] Filtering target genres and processing ratings in chunks...\n")
ratings_path = "../data/movielens32m/ratings.csv"

# 1. Define the 7 specific genres of interest
selected_genres = ['Action', 'Adventure', 'Comedy', 'Drama', 'Romance', 'Sci-Fi', 'Thriller']
print(f"Target Genres ({len(selected_genres)}): {selected_genres}")

# 2. Filter out movies with no genres listed
valid_movies = movies_df[movies_df['genres'] != '(no genres listed)'][['movieId', 'genres']].copy()

# 3. Explode pipe-delimited genres
valid_movies['genre_list'] = valid_movies['genres'].str.split('|')
movies_exploded = valid_movies.explode('genre_list').rename(columns={'genre_list': 'genre'})

# 4. CRITICAL FILTER: Keep ONLY the 7 selected genres
# Any movie or genre outside this list is dropped before the merge
movies_exploded = movies_exploded[movies_exploded['genre'].isin(selected_genres)].copy()
print(f"Pre-filtered exploded mapping size: {len(movies_exploded):,} rows.\n")

# Dictionaries to aggregate counts
monthly_genre_counts = {}  # {(year_month, genre): count}
total_7genre_monthly = {}  # {year_month: total_count_across_7_genres}

chunk_size = 2_000_000
chunks_processed = 0

dtypes = {'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int64'}

for chunk in pd.read_csv(ratings_path, usecols=['movieId', 'timestamp'], dtype=dtypes, chunksize=chunk_size):
    # Convert epoch seconds to year-month period string (YYYY-MM)
    chunk['timestamp'] = pd.to_datetime(chunk['timestamp'], unit='s')
    chunk['year_month'] = chunk['timestamp'].dt.to_period('M')
    
    # Inner merge with the 7-genre filtered dataset
    merged = chunk.merge(movies_exploded, on='movieId', how='inner')
    
    # Aggregate counts per (year_month, genre)
    counts = merged.groupby(['year_month', 'genre']).size()
    for (ym, genre), cnt in counts.items():
        monthly_genre_counts[(ym, genre)] = monthly_genre_counts.get((ym, genre), 0) + cnt
        # Accumulates the sum ONLY across the 7 target genres for this month
        total_7genre_monthly[ym] = total_7genre_monthly.get(ym, 0) + cnt

    chunks_processed += 1
    print(f"Processed chunk {chunks_processed} ({chunks_processed * chunk_size:,} records checked)")


[2/5] Filtering target genres and processing ratings in chunks...

Target Genres (7): ['Action', 'Adventure', 'Comedy', 'Drama', 'Romance', 'Sci-Fi', 'Thriller']
Pre-filtered exploded mapping size: 99,468 rows.

Processed chunk 1 (2,000,000 records checked)
Processed chunk 2 (4,000,000 records checked)
Processed chunk 3 (6,000,000 records checked)
Processed chunk 4 (8,000,000 records checked)
Processed chunk 5 (10,000,000 records checked)
Processed chunk 6 (12,000,000 records checked)
Processed chunk 7 (14,000,000 records checked)
Processed chunk 8 (16,000,000 records checked)
Processed chunk 9 (18,000,000 records checked)
Processed chunk 10 (20,000,000 records checked)
Processed chunk 11 (22,000,000 records checked)
Processed chunk 12 (24,000,000 records checked)
Processed chunk 13 (26,000,000 records checked)
Processed chunk 14 (28,000,000 records checked)
Processed chunk 15 (30,000,000 records checked)
Processed chunk 16 (32,000,000 records checked)
Processed chunk 17 (34,000,000 r

<div style="
background:#1F2937;
padding:12px;
border-left:8px solid #f63b73;
border-radius:8px;
color:white;
font-size:24px;
font-weight:bold;">

🪄 Section 3: Preprocessing the loaded data

</div>

In [23]:
# constructing full time series dataframe
print("\n[3/5] Constructing normalized time series matrix for the 7 genres...\n")
records = []
for (ym, genre), cnt in monthly_genre_counts.items():
    records.append({'year_month': ym, 'genre': genre, 'rating_count': cnt, 'total_7genre_ratings': total_7genre_monthly[ym]})

ts_df = pd.DataFrame(records)


[3/5] Constructing normalized time series matrix for the 7 genres...



`Monthly genre popularity share is measured as the percentage of total monthly MovieLens rating activity associated with movies belonging to that genre.`

genre share (%) = (total ratings of genre/total ratings) x 100 (for a month)

In [24]:
# Compute Relative Market Share (%) strictly among the 7 genres
ts_df['genre_share_pct'] = (ts_df['rating_count'] / ts_df['total_7genre_ratings']) * 100.0

# Convert year_month to datetime timestamp (Month Start: 'YYYY-MM-01')
ts_df['date'] = ts_df['year_month'].dt.to_timestamp()
ts_df = ts_df.sort_values(['date', 'genre']).reset_index(drop=True)

# For any given month: Sum of genre_share_pct across all 7 genres == 100.0%
# Verification check: verify that each month sums to exactly 100%
# monthly_sum_check = ts_df.groupby('date')['genre_share_pct'].sum()
# print(f"Verification: Mean monthly sum across the 7 genres = {monthly_sum_check.mean():.2f}% (Std: {monthly_sum_check.std():.4f})")

<div style="
background:#1F2937;
padding:12px;
border-left:8px solid #f63b73;
border-radius:8px;
color:white;
font-size:24px;
font-weight:bold;">

🗃️ Section 4: Filtering and Generating the Time series Data

</div>

In [27]:
# Filter date range
ts_df = ts_df[(ts_df['date'] >= '1996-01-01') & (ts_df['date'] <= '2022-12-31')]

# 1. Pivot raw rating counts (convert to integers since counts are whole numbers)
pivot_counts = (ts_df.pivot(index='date', columns='genre', values='rating_count').fillna(0).astype(int))

# 2. Pivot relative market share percentages
pivot_share = ts_df.pivot(index='date', columns='genre', values='genre_share_pct').fillna(0)

print(f'Successfully generated time series spanning {len(pivot_counts)} monthly' f' observations.')

Successfully generated time series spanning 323 monthly observations.


<div style="
background:#1F2937;
padding:12px;
border-left:8px solid #3bf65a;
border-radius:8px;
color:white;
font-size:24px;
font-weight:bold;">

📥 Section 5: Exporting the Experimental Time Series Data

</div>

In [28]:
print("\n[5/5] Exporting extracted time series datasets...")
pivot_counts.to_csv("../data/processed/monthly_genre_raw_counts.csv")
pivot_share.to_csv("../data/processed/monthly_genre_market_share.csv")
ts_df.to_csv("../data/processed/full_time_series.csv")
print("Saved:")
print(" - monthly_genre_raw_counts.csv")
print(" - monthly_genre_market_share.csv")
print(" - full_time_series.csv")


[5/5] Exporting extracted time series datasets...
Saved:
 - monthly_genre_raw_counts.csv
 - monthly_genre_market_share.csv
 - full_time_series.csv


In [29]:
ts_df.head()

,year_month,genre,rating_count,total_7genre_ratings,genre_share_pct,date
4,1996-01,Action,5,83,6.024096,1996-01-01
5,1996-01,Adventure,7,83,8.433735,1996-01-01
6,1996-01,Comedy,21,83,25.301205,1996-01-01
7,1996-01,Drama,18,83,21.686747,1996-01-01
8,1996-01,Romance,10,83,12.048193,1996-01-01


In [30]:
pivot_counts.head()

genre,Action,Adventure,Comedy,Drama,Romance,Sci-Fi,Thriller
date,,,,,,,
1996-01-01,5,7,21,18,10,4,18
1996-02-01,75,58,160,242,127,21,126
1996-03-01,2500,1840,3403,4507,2013,1010,2486
1996-04-01,15024,11492,19084,23688,10656,5545,15133
1996-05-01,54846,40507,65410,79643,36452,20578,59727


In [31]:
pivot_share.head()

genre,Action,Adventure,Comedy,Drama,Romance,Sci-Fi,Thriller
date,,,,,,,
1996-01-01,6.024096,8.433735,25.301205,21.686747,12.048193,4.819277,21.686747
1996-02-01,9.270705,7.169345,19.777503,29.913473,15.698393,2.595797,15.574784
1996-03-01,14.077369,10.360944,19.162115,25.378681,11.335098,5.687257,13.998536
1996-04-01,14.931128,11.420962,18.966031,23.541571,10.590129,5.510723,15.039455
1996-05-01,15.356014,11.341320,18.313767,22.298782,10.205984,5.761515,16.722617


In [18]:
#(ts_df['rating_count'].groupby(ts_df['genre']).sum()/1e3).sort_values().plot.barh(figsize=(7, 5), title="Total ratings per genre (thousands)")
#plt.show()